# S9 - Observabilidad con Grafana y costos

**Actividad:** instrumentar el pipeline de S8 (`atmos-eventos`) con campos de observabilidad calculados en Spark (`isValid`, `processedAt`, `latencyMs`), conectar Kafka a Prometheus, armar un tablero real en Grafana, calcular latencia agregada por ventana de tiempo, y proponer alertas y una estimacion de costos basadas en datos reales.

## Objetivo

1. Reanudar Kafka y el productor/consumidor de S7, y levantar Prometheus + Grafana unidos a la misma red.
2. Crear la `SparkSession` con el conector de Kafka (el mismo arranque de S8).
3. Leer `atmos-eventos` y calcular tres columnas nuevas: `isValid`, `processedAt`, `latencyMs`.
4. Ver los eventos en consola, con sus campos de observabilidad.
5. Revisar throughput y lag con `lastProgress`.
6. Persistir la evidencia observable en Parquet, en paralelo a la consulta de consola.

## 3.1 Reanudar el entorno y levantar observabilidad

**Producto del paso:** Kafka, `kafka-exporter`, Kafka UI y el productor de sensores de S7 corriendo, mas Prometheus y Grafana unidos a la red de Kafka.

Este notebook reutiliza exactamente el mismo entorno de S8. Antes de continuar, en una terminal (fuera del notebook):

```bash
cd kafka
docker compose up -d
cd ../uso-atmos
docker compose up -d
docker exec -it lambda26-uso-atmos python /app/consumer_sensores.py
```

Sin -d: el consumidor arranca pegado a esta terminal para que veas en vivo que los eventos siguen llegando bien formados antes de levantar el resto. En cuanto veas pasar algunas lineas con "status": "consumed", detenlo con Ctrl+C - ya cumplio su funcion. Abre una pestana nueva de terminal para el productor:

```bash
cd uso-atmos
docker exec -it lambda26-uso-atmos python /app/producer_sensores.py
```

El productor corre sin -d, pegado a esta terminal - dejalo abierto y visible toda la sesion, igual que en S8. Abre una tercera pestana para el resto de los comandos de este paso.

kafka-exporter ya viene definido en kafka/compose.yml desde S6 - el docker compose up -d de arriba ya lo levanto. Verificalo:

```bash
docker ps --filter "name=lambda26-kafka-exporter"
```

Ahora crea la carpeta obs/ (todavia no existe en este repo) con Prometheus y Grafana:

**obs/compose.yml:**

```yaml
name: lambda26-obs

services:
  prometheus:
    image: prom/prometheus:v3.14.0
    container_name: lambda26-prometheus
    restart: unless-stopped
    ports:
      - "49090:9090"
    volumes:
      - ./prometheus/prometheus.yml:/etc/prometheus/prometheus.yml:ro
      - lambda26_prometheus_data:/prometheus
    networks:
      - lambda26-kafka-net

  grafana:
    image: grafana/grafana:11.4.0
    container_name: lambda26-grafana
    restart: unless-stopped
    ports:
      - "43000:3000"
    environment:
      GF_SECURITY_ADMIN_USER: admin
      GF_SECURITY_ADMIN_PASSWORD: admin
    volumes:
      - ./grafana/provisioning:/etc/grafana/provisioning:ro
      - lambda26_grafana_data:/var/lib/grafana
    depends_on:
      - prometheus
    networks:
      - lambda26-kafka-net

volumes:
  lambda26_prometheus_data:
  lambda26_grafana_data:

networks:
  lambda26-kafka-net:
    external: true
    name: lambda26-kafka-net
```

**obs/prometheus/prometheus.yml:**

```yaml
global:
  scrape_interval: 15s
  evaluation_interval: 15s

scrape_configs:
  - job_name: prometheus
    static_configs:
      - targets:
          - localhost:9090

  - job_name: kafka-exporter
    metrics_path: /metrics
    static_configs:
      - targets:
          - kafka-exporter:9308
```

**obs/grafana/provisioning/datasources/datasources.yml:**

```yaml
apiVersion: 1

datasources:
  - name: Prometheus
    type: prometheus
    access: proxy
    url: http://prometheus:9090
    isDefault: true
```

Subelo:

```bash
cd ../obs
docker compose up -d
```

Error frecuente: si Grafana no encuentra el datasource de Prometheus, confirma que el contenedor lambda26-prometheus este corriendo (docker ps) y que la URL del datasource sea http://prometheus:9090 - el nombre del servicio Docker, no localhost.

## 3.4 Crear el notebook y la SparkSession, con el conector de Kafka

**Producto del paso:** notebook `09_observabilidad_pipeline_grafana_costos.ipynb` con una SparkSession capaz de leer y escribir Kafka - el mismo arranque que ya hiciste en S8.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion9-observabilidad")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "3")
    .config("spark.ui.port", "4040")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark

## 3.5 Leer el topic y calcular los campos de observabilidad

**Producto del paso:** un DataFrame con los campos de atmos-eventos ya tipados, mas tres columnas nuevas que no existian en S8: isValid, processedAt y latencyMs.

In [ ]:
from pyspark.sql.functions import col, from_json, current_timestamp, unix_millis
from pyspark.sql.types import StructType, StringType, DoubleType, LongType

esquema_evento = (
    StructType()
    .add("tipoEvento", StringType())
    .add("sensorId", StringType())
    .add("temperatura", DoubleType())
    .add("humedad", DoubleType())
    .add("presion", DoubleType())
    .add("origen", StringType())
    .add("timestamp", LongType())
)

crudo = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "atmos-eventos")
    .option("startingOffsets", "latest")
    .load()
)

eventos = (
    crudo.select(
        col("topic"), col("partition"), col("offset"),
        col("timestamp").alias("kafkaTimestamp"),
        col("value").cast("string").alias("json_str"),
    )
    .select(
        "topic", "partition", "offset", "kafkaTimestamp",
        from_json(col("json_str"), esquema_evento).alias("e"),
    )
    .select("topic", "partition", "offset", "kafkaTimestamp", "e.*")
)

observables = (
    eventos
    .withColumn(
        "isValid",
        col("tipoEvento").isNotNull()
        & col("sensorId").isNotNull()
        & col("temperatura").isNotNull()
        & col("timestamp").isNotNull(),
    )
    .withColumn("processedAt", unix_millis(current_timestamp()))
    .withColumn("latencyMs", col("processedAt") - col("timestamp"))
)
observables.printSchema()

kafkaTimestamp (metadata de Kafka: cuando el broker recibio el mensaje) es distinto de timestamp (un campo del propio evento: cuando el sensor midio, igual que en S8). processedAt es el momento en que esta celda de Spark proceso el evento - y latencyMs es la diferencia entre ambos: cuanto tardo un evento en viajar de producer_sensores.py, por Kafka, hasta este punto del pipeline. isValid reusa la misma idea de contrato minimo que ya viste en S7 (consumer_sensores.py), pero calculada en Spark en vez de en Python simple.

## 3.6 Ver los eventos en consola, con sus campos de observabilidad

**Producto del paso:** la salida de consola de S8, ahora con isValid, processedAt y latencyMs visibles fila por fila.

In [ ]:
# Momento 1: arrancar la consulta
consulta = (
    observables.writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", "false")
    .trigger(processingTime="5 seconds")
    .start()
)

Dejala corriendo - no la detengas todavia. Vas a usarla en 3.7 para revisar lastProgress, y en 3.8 vas a arrancar una segunda consulta que persiste en Parquet al mismo tiempo. Las vas a detener juntas al final de 3.8.

## 3.7 Revisar throughput y lag con lastProgress

**Producto del paso:** una lectura real de consulta.lastProgress, con el throughput del ultimo micro-lote y el offset de cada particion de Kafka.

In [ ]:
import json

progreso = consulta.lastProgress

if progreso is None:
    print("Todavia no hay micro-lote procesado. Espera unos segundos y corre esta celda de nuevo.")
else:
    fuente = progreso["sources"][0]
    metricas = fuente.get("metrics", {})
    estado = "idle" if progreso["numInputRows"] == 0 else "processed"

    log = {
        "service": "spark-streaming",
        "component": "observabilidad",
        "topic": "atmos-eventos",
        "batchId": progreso["batchId"],
        "numInputRows": progreso["numInputRows"],
        "inputRowsPerSecond": progreso["inputRowsPerSecond"],
        "processedRowsPerSecond": progreso["processedRowsPerSecond"],
        "avgOffsetsBehindLatest": metricas.get("avgOffsetsBehindLatest"),
        "maxOffsetsBehindLatest": metricas.get("maxOffsetsBehindLatest"),
        "status": estado,
    }
    print(json.dumps(log, indent=2))

numInputRows es cuantos eventos trajo el ultimo micro-lote; inputRowsPerSecond/processedRowsPerSecond son la velocidad de entrada y de procesamiento - si processedRowsPerSecond es mayor que inputRowsPerSecond, Spark esta procesando mas rapido de lo que llegan los datos, que es la situacion sana. avgOffsetsBehindLatest y maxOffsetsBehindLatest son el lag visto desde el lado de Spark: cuanto offset de Kafka le falta por leer a esta consulta en este momento. Si numInputRows sale 0, no es un error - significa que ese micro-lote en particular no encontro eventos nuevos (status: "idle").

Evidencia alternativa - Spark UI: abre http://localhost:4040 y entra a Structured Streaming -> Streaming Query Statistics. Ahi ves los mismos numeros (Input Rate, Process Rate, Batch Duration) de forma grafica, sin tener que leer lastProgress a mano.

## 3.8 Persistir la evidencia observable en Parquet

**Producto del paso:** los eventos con sus campos de observabilidad aterrizando en disco, al mismo tiempo que la consulta de consola de 3.6 sigue corriendo.

In [ ]:
# Momento 1: arrancar el stream a Parquet (la consulta de 3.6 sigue corriendo en paralelo)
ARTIFACTS = "./artifacts/atmos_eventos_observabilidad"
CHECKPOINT_PARQUET = "./artifacts/chk_observabilidad"

consulta2 = (
    observables.writeStream
    .outputMode("append")
    .format("parquet")
    .option("path", ARTIFACTS)
    .option("checkpointLocation", CHECKPOINT_PARQUET)
    .trigger(processingTime="5 seconds")
    .start()
)

Nota el nombre consulta2, no consulta: la consulta de 3.6 sigue viva en la variable consulta, y si esta nueva reusara ese mismo nombre, perderias la unica referencia que te permite detenerla mas tarde.

Mientras corre, puedes leer lo que ya escribio sin detenerla - este paso es opcional:

In [ ]:
# Momento 2 (opcional): leer lo guardado (se puede repetir, el stream sigue corriendo)
guardado = spark.read.parquet(ARTIFACTS)
total = guardado.count()
print("filas guardadas:", total)
guardado.select("sensorId", "latencyMs", "isValid").show(10, truncate=False)

Cuando ya viste suficiente, deten las dos consultas juntas - la de consola de 3.6 y la de Parquet de este paso:

In [ ]:
# Momento 3: detener ambas consultas (la de consola de 3.6 y la de Parquet de 3.8)
consulta.stop()
consulta2.stop()

Error frecuente: si el log muestra lineas ERROR WriteToDataSourceV2Exec, TaskKilledException o Aborting task al llamar .stop(), es el mismo ruido benigno que ya viste en S8 - Spark cancelando una tarea a mitad de un micro-lote, no una falla real.